In [ ]:
# === Cell 1: Environment configuration (Kaggle P100 + optional Windows GV100) ===

import torch

# ---------- WINDOWS + GV100 (LOCAL, COMMENTED ON KAGGLE) ----------
# Uncomment this block on your Windows machine and comment the Kaggle block below.
# DATA_DIR = r"C:\Users\GPU\sign_language\original_images"  # adjust if needed
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# NUM_WORKERS = 0   # if you see DataLoader issues on Windows, set this to 0


# ---------- KAGGLE + P100 (ACTIVE HERE) ----------
# Use this on Kaggle
DATA_DIR = "/kaggle/input/datasets/vrajesh0sharma7/sign-language/isl_dataset/custom_images"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 4  # Kaggle: 2–4 usually fine; increase if you want

print("Using device:", device)
print("DATA_DIR:", DATA_DIR)

In [ ]:
# === Cell 2: Imports, hyperparameters, and plotting style ===

import os
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

from sklearn.metrics import classification_report, confusion_matrix

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

IMG_SIZE = 224
BATCH_SIZE = 64
VAL_SPLIT = 0.2
NUM_EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7
LABEL_SMOOTHING = 0.1

# Plot style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

In [ ]:
# === Cell 3: Dataset loading (ImageFolder) + train/val split ===

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)

print("Total images:", len(full_dataset))
print("Number of classes:", num_classes)
print("Classes:", class_names)

# train/val split
n_total = len(full_dataset)
n_val = int(VAL_SPLIT * n_total)
n_train = n_total - n_val

train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val])

# assign transforms explicitly
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Train size:", len(train_dataset), "Val size:", len(val_dataset))

In [ ]:
# === Cell 4: EDA – per-class image counts and distribution plots ===

train_indices = train_dataset.indices
val_indices = val_dataset.indices

overall_counts = Counter([full_dataset.targets[i] for i in range(len(full_dataset))])
train_counts   = Counter([full_dataset.targets[i] for i in train_indices])
val_counts     = Counter([full_dataset.targets[i] for i in val_indices])

rows = []
for idx, name in enumerate(class_names):
    rows.append({
        "class_idx": idx,
        "class_name": name,
        "overall": overall_counts[idx],
        "train": train_counts[idx],
        "val": val_counts[idx],
    })

counts_df = pd.DataFrame(rows)
display(counts_df)

# overall images per class
plt.figure(figsize=(14, 6))
sns.barplot(data=counts_df, x="class_name", y="overall", color="skyblue")
plt.title("Overall Images per Class")
plt.xlabel("Class")
plt.ylabel("Image Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# train vs val per class
plt.figure(figsize=(14, 6))
x = np.arange(len(class_names))
bar_width = 0.4

plt.bar(x - bar_width/2, counts_df["train"], width=bar_width,
        label="Train", color="tab:blue")
plt.bar(x + bar_width/2, counts_df["val"],   width=bar_width,
        label="Val",   color="tab:orange")

plt.title("Train vs Validation Images per Class")
plt.xlabel("Class")
plt.ylabel("Image Count")
plt.xticks(x, class_names, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 5: EDA – random sample images from training set ===

def denormalize(img_tensor):
    img = img_tensor.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = std * img + mean
    img  = np.clip(img, 0, 1)
    return img

inputs, labels = next(iter(train_loader))

plt.figure(figsize=(10, 10))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    img = denormalize(inputs[i].cpu())
    plt.imshow(img)
    plt.title(class_names[labels[i].item()])
    plt.axis("off")

plt.suptitle("Random Training Samples", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 6: Baseline CNN architectures (from scratch) ===

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class SimpleCNNDropout(nn.Module):
    def __init__(self, num_classes, p=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(p),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# === Cell 7: Transfer-learning models (ResNet18, MobileNetV2, EfficientNet-B0, VGG16) ===

from torchvision.models import (
    ResNet18_Weights,
    MobileNet_V2_Weights,
    EfficientNet_B0_Weights,
    VGG16_Weights,
)

def create_model(arch: str, num_classes: int, pretrained: bool = True, dropout_p: float = 0.5):
    arch = arch.lower()

    if arch == "cnn":
        model = SimpleCNN(num_classes)

    elif arch == "cnn_dropout":
        model = SimpleCNNDropout(num_classes, p=dropout_p)

    elif arch == "resnet18":
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)  # ResNet18 pretrained
        for p in model.parameters():
            p.requires_grad = True
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(dropout_p),
            nn.Linear(in_features, num_classes)
        )

    elif arch == "mobilenet_v2":
        weights = MobileNet_V2_Weights.DEFAULT if pretrained else None
        model = models.mobilenet_v2(weights=weights)  # MobileNetV2 pretrained
        for p in model.parameters():
            p.requires_grad = True
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif arch == "efficientnet_b0":
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)  # EfficientNet-B0 pretrained
        for p in model.parameters():
            p.requires_grad = True
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif arch == "vgg16":
        weights = VGG16_Weights.DEFAULT if pretrained else None
        model = models.vgg16(weights=weights)  # VGG16 pretrained
        for p in model.features.parameters():
            p.requires_grad = True
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unknown architecture '{arch}'")

    return model.to(device)

In [ ]:
# === Cell 8: Training and validation loops with tqdm bars ===

def train_one_epoch(model, loader, criterion, optimizer, epoch):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    loop = tqdm(loader, desc=f"Epoch {epoch} [Train]", unit="batch")
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        loop.set_postfix(loss=f"{epoch_loss:.4f}", acc=f"{epoch_acc:.4f}")

    return epoch_loss, epoch_acc


def eval_one_epoch(model, loader, criterion, epoch):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    loop = tqdm(loader, desc=f"Epoch {epoch} [Val]", unit="batch")
    with torch.no_grad():
        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            epoch_loss = running_loss / total
            epoch_acc = correct / total
            loop.set_postfix(loss=f"{epoch_loss:.4f}", acc=f"{epoch_acc:.4f}")

    return epoch_loss, epoch_acc

In [ ]:
# === Cell 9: Generic experiment runner for any architecture ===

def run_experiment(
    arch_name: str,
    num_epochs: int = NUM_EPOCHS,
    lr: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
    label_smoothing: float = LABEL_SMOOTHING,
    pretrained: bool = True
):
    print(f"\n\n=== Architecture: {arch_name} ===")

    model = create_model(arch_name, num_classes=num_classes,
                         pretrained=pretrained, dropout_p=0.5)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3,
    )

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": []
    }

    best_val_loss = float("inf")
    best_val_acc  = 0.0
    best_state    = None
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
        val_loss, val_acc     = eval_one_epoch(model, val_loader, criterion, epoch)
    
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"[{arch_name}] Epoch {epoch:02d}: "
              f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"[{arch_name}] Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_val_loss, best_val_acc

In [ ]:
# === Cell 10: Run all architectures and build summary table ===

architectures = [
    "cnn",
    "cnn_dropout",
    "resnet18",
    "mobilenet_v2",
    "efficientnet_b0",
    "vgg16",
]

all_models = {}
all_histories = {}
summary_results = []

for arch in architectures:
    pretrained = False if arch in ["cnn", "cnn_dropout"] else True

    model, history, best_val_loss, best_val_acc = run_experiment(
        arch_name=arch,
        num_epochs=3,
        pretrained=pretrained
    )

    all_models[arch] = model
    all_histories[arch] = history
    summary_results.append({
        "architecture": arch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc
    })

summary_df = pd.DataFrame(summary_results)
display(summary_df.sort_values("best_val_acc", ascending=False))

In [ ]:
# === Cell 11: Comparison plots (best validation accuracy & loss) ===

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.barplot(data=summary_df, x="architecture", y="best_val_acc", palette="viridis")
plt.title("Best Validation Accuracy per Architecture")
plt.ylim(0.0, 1.0)
plt.xlabel("Architecture")
plt.ylabel("Accuracy")
plt.xticks(rotation=45, ha="right")

plt.subplot(1, 2, 2)
sns.barplot(data=summary_df, x="architecture", y="best_val_loss", palette="magma")
plt.title("Best Validation Loss per Architecture")
plt.xlabel("Architecture")
plt.ylabel("Loss")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# === Cell 12: Overlay validation accuracy/loss for selected architectures ===

compare_archs = ["cnn", "cnn_dropout", "resnet18", "efficientnet_b0"]

plt.figure(figsize=(12, 4))

# Validation accuracy
plt.subplot(1, 2, 1)
for arch in compare_archs:
    hist = all_histories[arch]
    epochs_range = range(1, len(hist["val_acc"]) + 1)
    plt.plot(epochs_range, hist["val_acc"], label=arch)
plt.title("Validation Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Validation loss
plt.subplot(1, 2, 2)
for arch in compare_archs:
    hist = all_histories[arch]
    epochs_range = range(1, len(hist["val_loss"]) + 1)
    plt.plot(epochs_range, hist["val_loss"], label=arch)
plt.title("Validation Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# === Cell 13: Detailed evaluation for best model (classification report + confusion matrix) ===

best_row = summary_df.sort_values("best_val_acc", ascending=False).iloc[0]
best_arch = best_row["architecture"]
print("Best architecture by validation accuracy:", best_arch)

best_model = all_models[best_arch]
best_model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = best_model(inputs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report for", best_arch)
print(classification_report(
    all_labels, all_preds,
    target_names=class_names,
    digits=4
))

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix – {best_arch}")
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 14: Correct and misclassified examples for best model ===

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

correct_indices = np.where(all_preds == all_labels)[0]
incorrect_indices = np.where(all_preds != all_labels)[0]

def get_val_sample(idx):
    img, label = val_dataset[idx]
    return img, label

# Correctly classified
n_show = min(16, len(correct_indices))
plt.figure(figsize=(10, 10))
for i in range(n_show):
    ax = plt.subplot(4, 4, i + 1)
    idx = correct_indices[i]
    img, label = get_val_sample(idx)
    img_np = denormalize(img.cpu())
    plt.imshow(img_np)
    plt.title(f"T: {class_names[label]}\nP: {class_names[all_preds[idx]]}")
    plt.axis("off")
plt.suptitle(f"Correctly Classified Examples – {best_arch}", y=1.02)
plt.tight_layout()
plt.show()

# Misclassified
if len(incorrect_indices) > 0:
    n_show = min(16, len(incorrect_indices))
    plt.figure(figsize=(10, 10))
    for i in range(n_show):
        ax = plt.subplot(4, 4, i + 1)
        idx = incorrect_indices[i]
        img, label = get_val_sample(idx)
        img_np = denormalize(img.cpu())
        plt.imshow(img_np)
        plt.title(f"T: {class_names[label]}\nP: {class_names[all_preds[idx]]}", color="red")
        plt.axis("off")
    plt.suptitle(f"Misclassified Examples – {best_arch}", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples in validation set.")